# Testing XAI Methods Programmatically

Calling the XAI methods outside of the website, The `TrainedModelXAI` class in the `inference_bridge` is designed to be fully decoupled from the web backend.

This notebook demonstrates how to load the model and programmatically generate explanations for any text using:
1. **Integrated Gradients (IG)** (Aspect & Conflict driver detection)
2. **LIME** (Local Interpretable Model-agnostic Explanations)
3. **SHAP** (SHapley Additive exPlanations)

> **Note:** Using LIME and SHAP might require installing the respective packages (`pip install lime shap`).

In [ ]:
import os
import sys
import textwrap

# Setup paths to import from inference_bridge
ML_RESEARCH_DIR = os.path.abspath(os.path.join("..", ".."))
INFERENCE_BRIDGE_DIR = os.path.join(ML_RESEARCH_DIR, "inference_bridge")
SRC_DIR = os.path.join(ML_RESEARCH_DIR, "src")

if INFERENCE_BRIDGE_DIR not in sys.path:
    sys.path.insert(0, INFERENCE_BRIDGE_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Import the XAI Wrapper
from trained_model_xai import TrainedModelXAI

# Load the model checkpoint
CHECKPOINT_PATH = os.path.join(ML_RESEARCH_DIR, "outputs", "cosmetic_sentiment_v1", "best_model.pt")

print("Loading model...")
xai = TrainedModelXAI(CHECKPOINT_PATH, temperature=0.5)

## Helper Function to Print Tokens
We'll create a nice printing function to visualize the output.

In [ ]:
def print_explanation(title, result):
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    
    if "predicted" in result:
        print(f"Predicted Sentiment: {result['predicted'].upper()} (Confidence: {result['confidence']:.2%})")
        print(f"Probabilities:  Negative: {result['probs']['negative']:.2%}  |  Neutral: {result['probs']['neutral']:.2%}  |  Positive: {result['probs']['positive']:.2%}")
        print("-"*60)
        
    print(f"Top Influencing Tokens (Method: {result['method']}):")
    for token, score in result['top_tokens']:
        # Simple bar chart based on score magnitude
        bar_len = int(abs(score) * 20)
        bar = '█' * bar_len
        direction = "(POS effect)" if score > 0 else "(NEG effect)"
        
        print(f"  {token:<15} : {score:>8.4f} | {bar} {direction}")

## 1. Integrated Gradients
Fast, accurate gradient-based method. Let's analyze a mixed review.

In [ ]:
text = "The colour is absolutely gorgeous but unfortunately the smell is terrible and overwhelming."

print(f"REVIEW TEXT:\n{text}\n")

ig_colour = xai.explain_ig_aspect(text, aspect="colour")
print_explanation("IG Explanation for 'colour'", ig_colour)

ig_smell = xai.explain_ig_aspect(text, aspect="smell")
print_explanation("IG Explanation for 'smell'", ig_smell)

## 2. Detecting Conflict Drivers
Find words that drive mixed sentiment across the review simultaneously.

In [ ]:
conflict_res = xai.explain_ig_conflict(text)
print_explanation("Conflict Drivers (tokens active for BOTH positive and negative aspects)", conflict_res)

## 3. LIME Explanation
Uses perturbation. It might take a few seconds.

In [ ]:
# Requires: pip install lime
try:
    lime_res = xai.explain_lime_aspect(text, aspect="smell", num_samples=100)
    print_explanation("LIME Explanation for 'smell'", lime_res)
except ImportError:
    print("LIME is not installed. Run `pip install lime` to try this method.")

## 4. SHAP Explanation
Uses cooperative game theory. Typically slower but mathematically robust.

In [ ]:
# Requires: pip install shap
try:
    shap_res = xai.explain_shap_aspect(text, aspect="colour", max_evals=100)
    print_explanation("SHAP Explanation for 'colour'", shap_res)
except ImportError:
    print("SHAP is not installed. Run `pip install shap` to try this method.")